<span style="font-size:20px">Comparative Analysis of KNN and Naive Bayes for Income Level Classificatoin</span>


Author: Gerardo Parra Diaz

Date: 3/1/2026

Course: DAT-402

<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Introduction
</h1>

The Adult Income dataset is a collection of 1994 U.S. Census records used to predict whether an individual earns more than $50,000 per year. It consists of 14 demographic features, including a mix of numerical data like age and categorical data like education or occupation.

The primary objective of this project is to develop and evaluate machine learnig models capablity of predicting wether an individuals annual income exceeds $50,000 based on 1994 US Census Bureau data. This is a binary classificatoin problem involving a mix of continous (numerical) and categorical (string-based) features.



The I got the dataset from the UC Irive Machine Learning Repository website(https://archive.ics.uci.edu/dataset/2/adult)

**Research Question:** To what extent can K-Nearest Neighbors (KNN) and Naive Bayes classifiers accurately predict an individual's income level (above or below $50,000) using 1994 Census data, and how do their underlying mathematical assumptions handle high-dimensional, skewed feature sets?

In [1]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report


<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Importing Dataset
</h1>

In [2]:

# Load it directly
df = pd.read_csv("adult.data")

df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'adult.data'

<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Data Wrangaling
</h1>


Using the adults.names file that came with the datset zip file, decsripotins were given for each column, which is what I used to rename the column names to the apportaivte variable.

In [ ]:
#Create column names since file does not have a header
col_names = ["age", "workclass", "fnlwgt", "education", "education-num", 
           "marital-status", "occupation", "relationship", "race", "sex", 
           "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"]

df = pd.read_csv("adult.data", names=col_names)

df.head()

Researching the dataset before importing it into Jupyter Notebook, I found that this census bureau dataset is notorious for having whitespaces in some of the entries and '?' for na values. That will cause a problem for some of the code later so I will first see if there are any whitespaces in the dataset.

In [ ]:
#this prints the first value of 'workclass' inside brackets to reveal spaces
print(f"'{df['workclass'].iloc[0]}'")

Since the output was ' state-gove' and not 'state-gove' there is whitespace that Pandas is triming for readablity. I wll strip these whitespaces.

In [ ]:
df = df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print(f"'{df['workclass'].iloc[0]}'")


Now for the na values, this dataset uses question marks to indicate if an entry was not avaliable.

In [ ]:
df.isin(['?']).sum()

As we can see 3 columns contian hundreds or thousands of these question mark entries. We will drop them in the same manner as we would NA values but first we will convert these entries as numpy Na objects.

In [ ]:
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)

print(f"Rows after dropping: {len(df)}")


Since the dataset started with 32561 rows we know we dropped all right rows since we only have 30162 rows left.

Now we will encode the target variable, **income**. Right now it contains strings and ">50k".  In order to properly use Niave Bayes and KNN algorithms we need to convert this column into a numeric type variable.

In [ ]:


le = LabelEncoder()
df['income'] = le.fit_transform(df['income'])

df_final = pd.get_dummies(df, drop_first=True, dtype=int)

df_final.head()


<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Exploratory Data Analysis
</h1>

To understand the relationships within our data, we generated a Pearson correlation heatmap focusing on the numerical features and our target variable, **income**.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

The Naive Bayes classifier operates on the Strong Independence Assumption, meaning it assumes that the presence of one feature is unrelated to the presence of any other feature. Our heatmap revealed that several features (like education and occupation-related metrics) are indeed correlated.

Because $r = 0.34$ between education and income, the independence assumption is technically violated. This "feature dependency" is a primary reason why we expect the K-Nearest Neighbors (KNN) model—which does not rely on independence—to potentially outperform the Gaussian Naive Bayes model.

<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Scaling & Train/Test Split
</h1>

The features in this dataset exist on vastly different scales. For instance, age typically ranges from 17 to 90, while capital-gain can range from 0 to 99,999.

Because the K-Nearest Neighbors (KNN) algorithm relies on calculating the Euclidean Distance between data points, features with larger numerical ranges would disproportionately dominate the model’s decision-making.

To solve this, we applied StandardScaler to transform the data so that each feature has a mean ($\mu$) of 0 and a standard deviation ($\sigma$) of 1. This ensures that every feature—from education level to hours worked—contributes equally to the distance calculation.

In [ ]:
# 1. Separate features and target
X = df_final.drop('income', axis=1)
y = df_final['income']

# 2. Create the 4 variables: X_train, X_test, y_train, y_test
# we use 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

To accurately measure how well our models generalize to "new" data, we partitioned the dataset into a 80/20 train test split to fit the models

In [ ]:
#Traingin KNN
# Start with k=5
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
knn_preds = knn.predict(X_test_scaled)

In [ ]:
#Training Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_preds = nb.predict(X_test_scaled)

In this section, we evaluate the predictive accuracy of our classifiers and investigate the "performance gap" between KNN’s robust distance-based logic and Naive Bayes' sensitivity to data distribution.

In [ ]:
# KNN Predictions
knn_preds = knn.predict(X_test_scaled)

# Naive Bayes Predictions
nb_preds = nb.predict(X_test_scaled)

print(f"KNN Accuracy: {accuracy_score(y_test, knn_preds):.4f}")
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_preds):.4f}")



While KNN achieved a strong 81.32% accuracy, the Gaussian Naive Bayes model significantly underperformed at 45.02%

To move beyond simple accuracy scores, we utilize Confusion Matrices and Classification Reports to visualize where our models are making successful predictions and where they are failing.

In [ ]:
confusion_matrix(y_test, knn_preds)



In [ ]:
print("--- KNN Report ---")
print(classification_report(y_test, knn_preds))

print("\n--- Naive Bayes Report ---")
print(classification_report(y_test, nb_preds))

In [ ]:
# 1. Generate the raw matrix
cm = confusion_matrix(y_test, knn_preds)

# 2. Plot it
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['<=50K', '>50K'])
disp.plot(cmap='Blues')
plt.title('KNN Confusion Matrix')
plt.show()

The KNN model achieved a balanced performance with an accuracy of 81.32%. By examining its Confusion Matrix, we can see it effectively identifies the majority class (Low Income) while maintaining a respectable "Recall" for the minority class (High Income). Because KNN is distance-based, it is not "confused" by the non-normal distribution of features like capital gains.

The Classification Report for Gaussian NB reveals a critical failure: the model is performing worse than a random guess. In our dataset, most people have $0 for capital-gain, but a few have 99,999.

Gaussian NB attempts to fit these outliers into a "Bell Curve" (Normal Distribution), which stretches the mathematical probability so thin that the model loses its predictive power.

This proves that our features do not meet the Gaussian assumption, prompting us to test the Bernoulli Naive Bayes variant which is specifically designed for the binary (0/1) dummy variables we created during preprocessing.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt

# --- 1. Model Training & Testing ---
# Bernoulli NB
bnb = BernoulliNB()
bnb.fit(X_train_scaled, y_train)

# Get predictions and probabilities on TEST data
bnb_preds = bnb.predict(X_test_scaled)
bnb_probs = bnb.predict_proba(X_test_scaled)[:, 1]

# --- 2. Enhanced Evaluation ---
print(f"Bernoulli NB Test Accuracy: {accuracy_score(y_test, bnb_preds):.4f}")

# Confusion Matrix
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, bnb_preds, cmap='Blues', ax=ax[0])
ax[0].set_title('BernoulliNB Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, bnb_probs)
roc_auc = auc(fpr, tpr)
ax[1].plot(fpr, tpr, color='darkorange', label=f'ROC curve (AUC = {roc_auc:.2f})')
ax[1].plot([0, 1], [0, 1], color='navy', linestyle='--')
ax[1].set_title('Receiver Operating Characteristic')
ax[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

<h1 style="font-size:30px; color:#1f77b4; font-weight:bold;">
Conclusion
</h1>

The final results of this project demonstrate that K-Nearest Neighbors (KNN) and Bernoulli Naive Bayes are equally effective at predicting income levels, both achieving an accuracy of approximately 81.3%.

The most significant finding of this research was the failure of the Gaussian Naive Bayes model (45.02%), which highlighted the importance of matching an algorithm’s mathematical assumptions to the data's distribution

Improve predictive power, implementing ensemble methods like Random Forest would likely be more robust, as these models naturally handle the skewed distributions that challenged our Gaussian Naive Bayes approach. Overall, these refinements would transition the project from a baseline comparison to a highly tuned, production-ready classification system.
